# Task 03 — Baseline Model

**Agent:** `<claude / codex / antigravity>`  
**Date:** `<YYYY-MM-DD>`  
**Input:** `agents/<agent>/task_01/outputs/ingested.csv`

---

**Objective:** Train a baseline model with a proper evaluation harness. All preprocessing fitted on training data only.

> Before committing: **Kernel → Restart & Clear Output**

In [ ]:
# ── Imports & seed ──────────────────────────────────────────────────────────
import random
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression  # replace if regression task
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INPUT_PATH = Path("../task_01/outputs/ingested.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load & Split

In [ ]:
df = pd.read_csv(INPUT_PATH)

# TODO: set TARGET to the name of your target column
TARGET = "<target_column>"

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Split BEFORE any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## 2. Preprocessing (fitted on train only)

In [ ]:
# TODO: add/replace steps as appropriate for your dataset
# The Pipeline ensures fit() is called only on training data
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(random_state=SEED, max_iter=500))
])

## 3. Train

In [ ]:
# Select only numeric columns for this baseline
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()

pipeline.fit(X_train[numeric_cols], y_train)
print("Training complete")

## 4. Evaluate on Test Set

In [ ]:
y_pred  = pipeline.predict(X_test[numeric_cols])
y_proba = pipeline.predict_proba(X_test[numeric_cols])[:, 1] \
          if hasattr(pipeline.named_steps["model"], "predict_proba") else None

results = {
    "accuracy": round(accuracy_score(y_test, y_pred), 4),
    "f1":       round(f1_score(y_test, y_pred, average="weighted"), 4),
}
if y_proba is not None:
    results["auc"] = round(roc_auc_score(y_test, y_proba), 4)

for k, v in results.items():
    print(f"  {k:12s}: {v}")

## 5. Save Outputs

In [ ]:
# baseline_results.csv
results_df = pd.DataFrame([
    {"metric_name": k, "value": v} for k, v in results.items()
])
results_df.to_csv(OUTPUT_DIR / "baseline_results.csv", index=False)

# model.pkl
with open(OUTPUT_DIR / "model.pkl", "wb") as f:
    pickle.dump(pipeline, f)

# baseline_report.md
report = f"""# Baseline Model Report

**SEED:** {SEED}  
**Train/test split:** 80/20  
**Model:** {type(pipeline.named_steps['model']).__name__}  

## Preprocessing
<!-- TODO: describe every step -->
- StandardScaler fitted on training data only

## Results
{results_df.to_markdown(index=False)}

## Limitations
<!-- TODO: what does this baseline not capture? -->
"""
(OUTPUT_DIR / "baseline_report.md").write_text(report)
print("Saved: baseline_results.csv, model.pkl, baseline_report.md")